<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/06_labeling_states_FULL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB06_FULL — Derivação de Estados e Formulação Supervisionada por Célula

## 1. Objetivo da etapa

O NB06_FULL deriva os estados operacionais e constrói as bases supervisionadas de antecipação do **experimento FULL**. O processamento é realizado separadamente para cada `cell_id`, preservando a ordem temporal, a delimitação dos episódios e a independência entre as oito unidades experimentais.

A etapa recebe:

- os episódios críticos produzidos pelo NB04_FULL;
- os atributos temporais causais produzidos pelo NB05_FULL.

A saída organiza, por célula e por cenário, as janelas rotuladas, as bases supervisionadas, os diagnósticos metodológicos e os artefatos de rastreabilidade necessários às etapas posteriores.

## 2. Regra de processamento por célula

Cada célula é tratada como uma réplica independente. A rotulagem `NORMAL`, `BEFORE`, `DURING` e `AFTER`, bem como a construção do alvo prospectivo, ocorre dentro da própria série temporal da célula.

As células não são concatenadas antes da rotulagem. A consolidação `allcells` ocorre somente depois que os artefatos individuais foram produzidos e validados. Essa ordem impede que regiões temporais, episódios ou horizontes futuros atravessem fronteiras entre células.

## 3. Estados e alvo prospectivo

Para cada episódio crítico, o notebook atribui:

- `DURING` às janelas pertencentes ao episódio;
- `BEFORE` às até `K` janelas anteriores;
- `AFTER` às até `K` janelas posteriores;
- `NORMAL` às demais janelas.

Em caso de sobreposição, aplica-se a precedência:

```text
DURING > BEFORE > AFTER > NORMAL
```

O alvo supervisionado é positivo quando existe entrada em `DURING` dentro das próximas `H` janelas. A base supervisionada inclui apenas observações inicialmente em `NORMAL` ou `BEFORE` e remove as últimas `H` janelas, cujo horizonte futuro é incompleto.

## 4. Cenário principal e análises complementares

O cenário principal é:

```text
primary_K24_H12
```

com `K=24`, `H=12`, janelas de 5 minutos e horizonte prospectivo de 60 minutos.

Os demais cenários preservados nesta etapa permitem examinar a sensibilidade da formulação a diferentes relações entre contexto e horizonte. Cada artefato mantém `scenario_name` e `scenario_label`, evitando mistura entre cenários.

## 5. Rastreabilidade e denominadores

As tabelas por célula preservam explicitamente:

- quantidade de janelas rotuladas;
- distribuição dos estados;
- quantidade de amostras supervisionadas;
- número de casos positivos e negativos;
- `positive_rate` da base supervisionada de cada célula e cenário;
- quantidade de episódios efetivos;
- diagnósticos de equivalência ou degeneração entre `BEFORE` e o alvo.

A `positive_rate` registrada nesta etapa refere-se à proporção de `target=1` na base supervisionada correspondente. Ela não representa prevalência de treino ou de teste, pois o NB06_FULL não cria divisão treino–teste.

## 6. Validações metodológicas

A execução verifica, por célula e cenário:

- integridade temporal e unicidade das chaves;
- ausência de mistura entre células;
- consistência entre episódios e janelas `DURING`;
- existência de amostras supervisionadas;
- presença das classes positiva e negativa quando aplicável;
- não equivalência trivial entre o estado `BEFORE` e o alvo;
- presença de `BEFORE,target=0` no cenário principal;
- consistência dos aliases de cenário;
- equivalência entre artefatos individuais e consolidados.

As ocorrências metodológicas do cenário principal são registradas por célula em `06_FULL_primary_diagnostic_flags.csv`. O modo estrito pode interromper a execução quando `STRICT_PRIMARY_CHECKS=1`.

## 7. Artefatos produzidos

A etapa grava os arquivos sob:

```text
04-reports/99_FULL_downstream/06_FULL_labeling_states/
```

com separação funcional entre:

```text
features/
reports/
```

Os nomes dos arquivos preservam `cell_id` e cenário. Para o cenário principal, também são produzidos os agregados `allcells`, a tabela comparativa por célula, os summaries JSON e o manifesto SHA-256 com autorreferência controlada.

## 8. Responsabilidade sobre o split temporal

O NB06_FULL não define divisão treino–teste e não produz `n_positivos_test`. Essa contagem deve ser calculada pela etapa que efetivamente estabelece o protocolo temporal de modelagem, mantendo separados:

- a prevalência da base supervisionada;
- a prevalência do treino;
- a prevalência do teste fixo;
- as distribuições observadas nas partições progressivas.


In [ ]:

# ============================================================
# NB06_FULL — Derivação de Estados e Formulação Supervisionada por Célula
# Pipeline PPCOMP_DM — ramo FULL
#
# Escopo:
# - Ler as features por célula produzidas no NB05_FULL
# - Ler os episódios críticos oficiais por célula detectados no NB04_FULL
# - Ajustar episódios à faixa efetivamente disponível nas features de cada célula
# - Rotular janelas como NORMAL, BEFORE, DURING ou AFTER por célula
# - Construir bases supervisionadas para diferentes combinações de K e H por célula
# - Diagnosticar circularidade/degeneração entre BEFORE e alvo positivo
# - Persistir artefatos por célula, consolidados allcells, summaries JSON e manifesto SHA-256
#
# Decisão central:
# - Cada cell_id é uma réplica independente do experimento FULL.
# - As células só são concatenadas depois de rotuladas e validadas separadamente.
# ============================================================

from pathlib import Path
import os
import sys
import subprocess
import importlib
import random
import json
import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ─────────────────────────────────────────────────────────────
# BLOCO 0 — Bootstrap e configuração de diretórios
# ─────────────────────────────────────────────────────────────

def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


if running_in_colab() and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
elif running_in_colab():
    print("[Bootstrap] Google Drive já montado.")

# Diretório do repositório de código. Pode ser diferente da área de artefatos FULL.
DEFAULT_REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
REPO_DIR = Path(os.environ.get("PPCOMP_DM_ROOT", str(DEFAULT_REPO_DIR)))

# Diretório oficial dos artefatos do experimento FULL.
# No Windows: H:\Meu Drive\Mestrado\04-reports\99_FULL_downstream
DEFAULT_FULL_ROOT = Path("/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream")

GITHUB_REPO = os.environ.get(
    "PPCOMP_DM_GITHUB_REPO",
    "https://github.com/sergiocostaifes/PPCOMP_DM.git",
)

if running_in_colab():
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        print(f"[Bootstrap] Clonando repositório em: {REPO_DIR}")
        subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
    else:
        print(f"[Bootstrap] Repositório encontrado: {REPO_DIR}")
        if os.environ.get("PPCOMP_DM_SKIP_GIT_PULL", "0") != "1":
            try:
                print("[Bootstrap] Atualizando repositório (git pull)...")
                subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
            except Exception as e:
                print("[Bootstrap] Aviso: não foi possível atualizar via git pull:", e)

if REPO_DIR.exists():
    os.chdir(str(REPO_DIR))
    repo_str = str(REPO_DIR)
    if repo_str not in sys.path:
        sys.path.insert(0, repo_str)
    importlib.invalidate_caches()
    print("[Bootstrap] CWD =", os.getcwd())
else:
    print(f"[Bootstrap] Aviso: REPO_DIR não existe neste ambiente: {REPO_DIR}")

FULL_ROOT = Path(os.environ.get("FULL_ROOT", str(DEFAULT_FULL_ROOT)))

NB04_FULL_DIR = Path(os.environ.get("NB04_FULL_DIR", str(FULL_ROOT / "04_FULL_episodes")))
NB05_FULL_DIR = Path(os.environ.get("NB05_FULL_DIR", str(FULL_ROOT / "05_FULL_features")))
NB06_FULL_DIR = Path(os.environ.get("NB06_FULL_DIR", str(FULL_ROOT / "06_FULL_labeling_states")))

if running_in_colab() and not FULL_ROOT.exists():
    print(f"[Bootstrap] Aviso: FULL_ROOT ainda não existe: {FULL_ROOT}")
    print("[Bootstrap] Confirme se o caminho corresponde a MyDrive/Mestrado/04-reports/99_FULL_downstream ou defina os.environ['FULL_ROOT'] antes desta célula.")

OUT_FEATURES_DIR = NB06_FULL_DIR / "features"
OUT_REPORTS_DIR = NB06_FULL_DIR / "reports"

for d in [NB06_FULL_DIR, OUT_FEATURES_DIR, OUT_REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("FULL_ROOT      =", FULL_ROOT)
print("NB04_FULL_DIR  =", NB04_FULL_DIR)
print("NB05_FULL_DIR  =", NB05_FULL_DIR)
print("NB06_FULL_DIR  =", NB06_FULL_DIR)
print("OUT_FEATURES   =", OUT_FEATURES_DIR)
print("OUT_REPORTS    =", OUT_REPORTS_DIR)


def log(msg: str) -> None:
    """Imprime mensagens padronizadas desta etapa do pipeline."""
    print(f"[NB06_FULL_labeling_states] {msg}")

# ─────────────────────────────────────────────────────────────
# BLOCO 1 — Parâmetros científicos e operacionais
# ─────────────────────────────────────────────────────────────

WINDOW_MINUTES = int(os.environ.get("WINDOW_MINUTES", "5"))

STATE_NORMAL = "NORMAL"
STATE_BEFORE = "BEFORE"
STATE_DURING = "DURING"
STATE_AFTER = "AFTER"

STATE_PRECEDENCE = {
    STATE_NORMAL: 0,
    STATE_AFTER: 1,
    STATE_BEFORE: 2,
    STATE_DURING: 3,
}

PRIMARY_K = int(os.environ.get("PRIMARY_K", "24"))
PRIMARY_H = int(os.environ.get("PRIMARY_H", "12"))
PRIMARY_SCENARIO_NAME = f"primary_K{PRIMARY_K:02d}_H{PRIMARY_H:02d}"
PRIMARY_SCENARIO_LABEL = PRIMARY_SCENARIO_NAME

# Por padrão, diagnósticos metodológicos por célula são registrados como flags,
# não como asserts fatais. Ative STRICT_PRIMARY_CHECKS=1 para transformar esses
# achados em erro explícito de execução.
STRICT_PRIMARY_CHECKS = os.environ.get("STRICT_PRIMARY_CHECKS", "0").strip() == "1"

# Mantém o cenário histórico e os cenários de sensibilidade definidos para esta etapa.
SCENARIOS = [
    {"name": "legacy_K12_H12", "K": 12, "H": 12},
    {"name": "sensitivity_K12_H06", "K": 12, "H": 6},
    {"name": "sensitivity_K12_H03", "K": 12, "H": 3},
    {"name": PRIMARY_SCENARIO_NAME, "K": PRIMARY_K, "H": PRIMARY_H},
    {"name": "sensitivity_K24_H06", "K": 24, "H": 6},
    {"name": "sensitivity_K24_H03", "K": 24, "H": 3},
]

# Remove duplicatas caso PRIMARY_* coincida com um cenário já listado.
_seen_scenarios = set()
SCENARIOS = [
    s for s in SCENARIOS
    if not (s["name"] in _seen_scenarios or _seen_scenarios.add(s["name"]))
]

ACTIVE_CELLS_RAW = os.environ.get("ACTIVE_CELLS", "a,b,c,d,e,f,g,h").strip()

if ACTIVE_CELLS_RAW.lower() in {"all", "*", ""}:
    ACTIVE_CELLS = list("abcdefgh")
else:
    ACTIVE_CELLS = [
        c.strip().lower()
        for c in ACTIVE_CELLS_RAW.split(",")
        if c.strip()
    ]

assert ACTIVE_CELLS, "Nenhuma célula ativa definida em ACTIVE_CELLS."

log(f"Células ativas: {ACTIVE_CELLS}")
log(f"Cenário principal: {PRIMARY_SCENARIO_NAME}")
log(f"STRICT_PRIMARY_CHECKS: {STRICT_PRIMARY_CHECKS}")

# ─────────────────────────────────────────────────────────────
# BLOCO 2 — Localização robusta dos artefatos NB04_FULL/NB05_FULL
# ─────────────────────────────────────────────────────────────

def normalize_cell_id(value) -> str:
    """Normaliza cell_id para string minúscula sem espaços."""
    return str(value).strip().lower()


def short_path(path: Path) -> str:
    try:
        return str(path.relative_to(REPO_DIR))
    except Exception:
        return str(path)


def glob_sorted(base: Path, patterns: list[str]) -> list[Path]:
    matches = []
    if not base.exists():
        return matches
    for pattern in patterns:
        matches.extend(base.glob(pattern))
    unique = sorted(set(p for p in matches if p.is_file()))
    return unique


def path_has_cell_token(path: Path, cell_id: str) -> bool:
    """Verifica se o caminho parece pertencer à célula solicitada."""
    s = str(path).lower().replace("\\", "/")
    tokens = [
        f"cell_{cell_id}",
        f"cell-{cell_id}",
        f"cell={cell_id}",
        f"/{cell_id}/",
        f"_{cell_id}.parquet",
        f"-{cell_id}.parquet",
    ]
    return any(t in s for t in tokens)


def find_artifact_file(base_dir: Path, cell_id: str, role: str) -> Path | None:
    """
    Localiza arquivos por célula com tolerância a pequenas variações de nome.

    role='features':
      artefato do NB05_FULL contendo features por bucket/célula.

    role='episodes':
      artefato do NB04_FULL contendo episódios críticos por célula.
    """
    cell_id = normalize_cell_id(cell_id)

    if role == "features":
        patterns = [
            f"**/*features*cell_{cell_id}*.parquet",
            f"**/*features*cell-{cell_id}*.parquet",
            f"**/*features*cell={cell_id}*.parquet",
            f"**/cell_{cell_id}/*features*.parquet",
            f"**/cell-{cell_id}/*features*.parquet",
            f"**/cell={cell_id}/*features*.parquet",
            f"**/*window_5min_features*{cell_id}*.parquet",
        ]
        include_terms = ["features"]
        exclude_terms = ["summary", "manifest", "allcells", "labeled", "supervised", "transition"]
    elif role == "episodes":
        patterns = [
            f"**/*episodes*cell_{cell_id}*.parquet",
            f"**/*episodes*cell-{cell_id}*.parquet",
            f"**/*episodes*cell={cell_id}*.parquet",
            f"**/cell_{cell_id}/*episodes*.parquet",
            f"**/cell-{cell_id}/*episodes*.parquet",
            f"**/cell={cell_id}/*episodes*.parquet",
            f"**/*episodes*{cell_id}*.parquet",
        ]
        include_terms = ["episodes"]
        exclude_terms = ["summary", "manifest", "allcells", "labeled", "supervised", "transition"]
    else:
        raise ValueError(f"role inválido: {role}")

    candidates = glob_sorted(base_dir, patterns)

    filtered = []
    for p in candidates:
        name = p.name.lower()
        if not all(term in str(p).lower() for term in include_terms):
            continue
        if any(term in name for term in exclude_terms):
            continue
        if not path_has_cell_token(p, cell_id):
            continue
        filtered.append(p)

    if not filtered:
        return None

    # Prioriza nomes com FULL/cell explícito e caminhos mais curtos.
    def score(p: Path) -> tuple:
        s = str(p).lower()
        return (
            0 if "full" in s else 1,
            0 if f"cell_{cell_id}" in s else 1,
            len(str(p)),
            str(p),
        )

    return sorted(filtered, key=score)[0]


def find_allcells_file(base_dir: Path, role: str) -> Path | None:
    """
    Localiza arquivo agregado allcells para fallback.

    Prioriza nomes explícitos com allcells. Se não houver, aceita um parquet
    genérico do NB04/NB05 que contenha o termo esperado e não tenha marcador
    explícito de célula no nome. A filtragem por cell_id ainda será validada
    depois da leitura.
    """
    if role == "features":
        priority_patterns = ["**/*features*allcells*.parquet", "**/*allcells*features*.parquet"]
        generic_patterns = ["**/*features*.parquet", "**/*window_5min_features*.parquet"]
        include = "features"
    elif role == "episodes":
        priority_patterns = ["**/*episodes*allcells*.parquet", "**/*allcells*episodes*.parquet"]
        generic_patterns = ["**/*episodes*.parquet"]
        include = "episodes"
    else:
        raise ValueError(role)

    def clean(candidates: list[Path], allow_cell_tokens: bool) -> list[Path]:
        out = []
        for p in candidates:
            name = p.name.lower()
            full = str(p).lower().replace("\\", "/")
            if include not in name and include not in full:
                continue
            if any(term in name for term in ["summary", "manifest", "labeled", "supervised", "transition"]):
                continue
            if not allow_cell_tokens and any(token in full for token in ["cell_", "cell-", "cell="]):
                continue
            out.append(p)
        return sorted(set(out), key=lambda p: (0 if "full" in str(p).lower() else 1, len(str(p)), str(p)))

    priority = clean(glob_sorted(base_dir, priority_patterns), allow_cell_tokens=True)
    if priority:
        return priority[0]

    generic = clean(glob_sorted(base_dir, generic_patterns), allow_cell_tokens=False)
    return generic[0] if generic else None


ALLCELLS_FEATURES_FILE = find_allcells_file(NB05_FULL_DIR, "features")
ALLCELLS_EPISODES_FILE = find_allcells_file(NB04_FULL_DIR, "episodes")

if ALLCELLS_FEATURES_FILE:
    log(f"Fallback features allcells localizado: {short_path(ALLCELLS_FEATURES_FILE)}")
if ALLCELLS_EPISODES_FILE:
    log(f"Fallback episódios allcells localizado: {short_path(ALLCELLS_EPISODES_FILE)}")


def read_cell_features(cell_id: str) -> tuple[pd.DataFrame, str]:
    """Lê as features da célula, preferindo arquivo individual e usando allcells como fallback."""
    cell_id = normalize_cell_id(cell_id)
    path = find_artifact_file(NB05_FULL_DIR, cell_id, "features")

    if path is not None:
        df = pd.read_parquet(path)
        source = str(path)
    elif ALLCELLS_FEATURES_FILE is not None:
        df_all = pd.read_parquet(ALLCELLS_FEATURES_FILE)
        assert "cell_id" in df_all.columns, (
            f"Fallback allcells de features não contém cell_id: {ALLCELLS_FEATURES_FILE}"
        )
        df = df_all.loc[
            df_all["cell_id"].map(normalize_cell_id) == cell_id
        ].copy()
        source = str(ALLCELLS_FEATURES_FILE) + f"::cell_id={cell_id}"
    else:
        raise FileNotFoundError(
            f"Não encontrei features do NB05_FULL para cell_id={cell_id} em {NB05_FULL_DIR}"
        )

    if "cell_id" not in df.columns:
        df["cell_id"] = cell_id
    else:
        df["cell_id"] = df["cell_id"].map(normalize_cell_id)

    return df, source


def read_cell_episodes(cell_id: str) -> tuple[pd.DataFrame, str]:
    """Lê episódios da célula, preferindo arquivo individual e usando allcells como fallback."""
    cell_id = normalize_cell_id(cell_id)
    path = find_artifact_file(NB04_FULL_DIR, cell_id, "episodes")

    if path is not None:
        df = pd.read_parquet(path)
        source = str(path)
    elif ALLCELLS_EPISODES_FILE is not None:
        df_all = pd.read_parquet(ALLCELLS_EPISODES_FILE)
        assert "cell_id" in df_all.columns, (
            f"Fallback allcells de episódios não contém cell_id: {ALLCELLS_EPISODES_FILE}"
        )
        df = df_all.loc[
            df_all["cell_id"].map(normalize_cell_id) == cell_id
        ].copy()
        source = str(ALLCELLS_EPISODES_FILE) + f"::cell_id={cell_id}"
    else:
        raise FileNotFoundError(
            f"Não encontrei episódios do NB04_FULL para cell_id={cell_id} em {NB04_FULL_DIR}"
        )

    if "cell_id" not in df.columns:
        df["cell_id"] = cell_id
    else:
        df["cell_id"] = df["cell_id"].map(normalize_cell_id)

    return df, source

# ─────────────────────────────────────────────────────────────
# BLOCO 3 — Funções de validação, rotulagem e alvo supervisionado
# ─────────────────────────────────────────────────────────────

def require_columns(df: pd.DataFrame, required: set[str], label: str) -> None:
    missing = required - set(df.columns)
    assert not missing, f"Colunas ausentes em {label}: {sorted(missing)}"


def prepare_features_df(df: pd.DataFrame, cell_id: str) -> pd.DataFrame:
    """Normaliza e valida a base de features de uma célula."""
    cell_id = normalize_cell_id(cell_id)
    df = df.copy()

    require_columns(
        df,
        {"cell_id", "bucket_id", "bucket_start_us", "fail_rate", "is_critical"},
        f"features cell={cell_id}",
    )

    df["cell_id"] = df["cell_id"].map(normalize_cell_id)
    df = df.loc[df["cell_id"] == cell_id].copy()

    assert len(df) > 0, f"Features vazias após filtro cell_id={cell_id}"

    df = df.sort_values("bucket_id").reset_index(drop=True)

    assert df["bucket_id"].is_monotonic_increasing, (
        f"bucket_id não está ordenado em features cell={cell_id}"
    )
    assert df["bucket_id"].is_unique, (
        f"bucket_id duplicado em features cell={cell_id}"
    )

    bucket_diff = df["bucket_id"].diff().dropna()
    n_gaps = int((bucket_diff > 1).sum())

    assert n_gaps == 0, (
        f"A base de features deveria estar contínua após o trim inicial "
        f"em cell={cell_id}; gaps encontrados={n_gaps}"
    )

    df["cell_id"] = cell_id
    return df


def normalize_episode_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aceita variações comuns de nomes vindas do NB04_FULL e normaliza para:
    episode_id, start_bucket, end_bucket.
    """
    df = df.copy()

    rename_candidates = {
        "episode_start_bucket": "start_bucket",
        "episode_end_bucket": "end_bucket",
        "start_bucket_id": "start_bucket",
        "end_bucket_id": "end_bucket",
        "episode_start_bucket_id": "start_bucket",
        "episode_end_bucket_id": "end_bucket",
    }

    for old, new in rename_candidates.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    if "episode_id" not in df.columns:
        df["episode_id"] = np.arange(1, len(df) + 1, dtype="int64")

    return df


def prepare_episodes_df(df: pd.DataFrame, cell_id: str) -> pd.DataFrame:
    """Normaliza e valida episódios de uma célula."""
    cell_id = normalize_cell_id(cell_id)
    df = normalize_episode_columns(df)

    require_columns(
        df,
        {"cell_id", "episode_id", "start_bucket", "end_bucket"},
        f"episodes cell={cell_id}",
    )

    df["cell_id"] = df["cell_id"].map(normalize_cell_id)
    df = df.loc[df["cell_id"] == cell_id].copy()

    assert len(df) > 0, f"Episódios vazios após filtro cell_id={cell_id}"

    df = df.sort_values(["start_bucket", "end_bucket"]).reset_index(drop=True)
    df["cell_id"] = cell_id

    invalid = df.loc[df["end_bucket"] < df["start_bucket"]]
    assert len(invalid) == 0, (
        f"Episódios com end_bucket < start_bucket em cell={cell_id}: {len(invalid)}"
    )

    return df


def assign_state_with_precedence(
    state_array: np.ndarray,
    rank_array: np.ndarray,
    mask: np.ndarray,
    new_state: str,
) -> None:
    """Atribui estado apenas quando sua precedência supera a já registrada."""
    new_rank = STATE_PRECEDENCE[new_state]
    update_mask = mask & (new_rank > rank_array)

    state_array[update_mask] = new_state
    rank_array[update_mask] = new_rank


def build_effective_episodes(
    episodes_df: pd.DataFrame,
    feature_bucket_min: int,
    feature_bucket_max: int,
) -> pd.DataFrame:
    """Recorta episódios detectados à faixa de buckets disponível nas features."""
    rows = []

    for _, row in episodes_df.iterrows():
        original_start = int(row["start_bucket"])
        original_end = int(row["end_bucket"])

        effective_start = max(original_start, feature_bucket_min)
        effective_end = min(original_end, feature_bucket_max)

        if effective_start <= effective_end:
            new_row = row.to_dict()
            new_row["original_start_bucket"] = original_start
            new_row["original_end_bucket"] = original_end
            new_row["effective_start_bucket"] = effective_start
            new_row["effective_end_bucket"] = effective_end
            new_row["was_clipped_by_feature_range"] = (
                effective_start != original_start
                or effective_end != original_end
            )
            new_row["effective_duration_windows"] = int(
                effective_end - effective_start + 1
            )
            rows.append(new_row)

    effective_df = pd.DataFrame(rows)

    if len(effective_df) > 0:
        effective_df = effective_df.sort_values(
            ["cell_id", "effective_start_bucket", "effective_end_bucket"]
        ).reset_index(drop=True)

    return effective_df


def label_states_for_k(
    features_df: pd.DataFrame,
    effective_episodes_df: pd.DataFrame,
    k_before: int,
    k_after: int,
    bucket_min: int,
    bucket_max: int,
) -> pd.DataFrame:
    """Rotula cada bucket como NORMAL, BEFORE, DURING ou AFTER para uma célula."""
    df = features_df.copy().sort_values("bucket_id").reset_index(drop=True)
    bucket_values = df["bucket_id"].to_numpy()

    state_array = np.array([STATE_NORMAL] * len(df), dtype=object)
    rank_array = np.zeros(len(df), dtype=np.int8)

    for _, ep in effective_episodes_df.iterrows():
        start_b = int(ep["effective_start_bucket"])
        end_b = int(ep["effective_end_bucket"])

        before_start = max(bucket_min, start_b - k_before)
        before_end = start_b - 1

        after_start = end_b + 1
        after_end = min(bucket_max, end_b + k_after)

        if after_start <= after_end:
            mask_after = (
                (bucket_values >= after_start)
                & (bucket_values <= after_end)
            )
            assign_state_with_precedence(
                state_array=state_array,
                rank_array=rank_array,
                mask=mask_after,
                new_state=STATE_AFTER,
            )

        if before_start <= before_end:
            mask_before = (
                (bucket_values >= before_start)
                & (bucket_values <= before_end)
            )
            assign_state_with_precedence(
                state_array=state_array,
                rank_array=rank_array,
                mask=mask_before,
                new_state=STATE_BEFORE,
            )

        mask_during = (
            (bucket_values >= start_b)
            & (bucket_values <= end_b)
        )
        assign_state_with_precedence(
            state_array=state_array,
            rank_array=rank_array,
            mask=mask_during,
            new_state=STATE_DURING,
        )

    df["state"] = state_array
    df["state_rank"] = rank_array
    df["state_window_k"] = int(k_before)

    df["next_bucket_id"] = df["bucket_id"].shift(-1)
    df["next_state"] = df["state"].shift(-1)
    df["is_last_window"] = df["next_bucket_id"].isna()

    df["has_valid_next"] = (
        (~df["is_last_window"])
        & ((df["next_bucket_id"] - df["bucket_id"]) == 1)
    )

    df["transition"] = np.where(
        df["has_valid_next"],
        df["state"] + "->" + df["next_state"],
        pd.NA,
    )

    return df


def add_future_horizon_fields(
    labeled_df: pd.DataFrame,
    horizon_windows: int,
) -> pd.DataFrame:
    """Avalia, dentro da mesma célula, estados futuros até H janelas."""
    df = labeled_df.copy().sort_values(["cell_id", "bucket_id"]).reset_index(drop=True)

    result_parts = []

    for cell_id, g in df.groupby("cell_id", sort=True):
        g = g.copy().sort_values("bucket_id").reset_index(drop=True)
        state_by_bucket = dict(zip(g["bucket_id"], g["state"]))

        future_valid_flags = []
        future_has_during_flags = []
        future_all_normal_flags = []
        future_path_states = []
        future_first_during_offset = []

        for b in g["bucket_id"].tolist():
            future_states = []
            valid_horizon = True

            for h in range(1, horizon_windows + 1):
                next_b = b + h
                s = state_by_bucket.get(next_b, None)

                if s is None:
                    valid_horizon = False
                    break

                future_states.append(s)

            if not valid_horizon:
                future_valid_flags.append(False)
                future_has_during_flags.append(False)
                future_all_normal_flags.append(False)
                future_path_states.append(pd.NA)
                future_first_during_offset.append(pd.NA)
                continue

            has_during = any(s == STATE_DURING for s in future_states)
            all_normal = all(s == STATE_NORMAL for s in future_states)

            first_during = pd.NA
            if has_during:
                for idx, s in enumerate(future_states, start=1):
                    if s == STATE_DURING:
                        first_during = idx
                        break

            future_valid_flags.append(True)
            future_has_during_flags.append(has_during)
            future_all_normal_flags.append(all_normal)
            future_path_states.append(" | ".join(future_states))
            future_first_during_offset.append(first_during)

        g["future_horizon_valid"] = future_valid_flags
        g["future_has_during"] = future_has_during_flags
        g["future_all_normal"] = future_all_normal_flags
        g["future_path_states"] = future_path_states
        g["future_first_during_offset"] = future_first_during_offset
        g["target_horizon_windows"] = int(horizon_windows)
        g["prediction_horizon_minutes"] = int(horizon_windows * WINDOW_MINUTES)

        result_parts.append(g)

    return pd.concat(result_parts, ignore_index=True)


def build_supervised_dataset(
    labeled_df: pd.DataFrame,
    horizon_windows: int,
    scenario_name: str,
) -> pd.DataFrame:
    """Constrói a base supervisionada binária para antecipação de DURING em H janelas futuras."""
    df_h = add_future_horizon_fields(
        labeled_df=labeled_df,
        horizon_windows=horizon_windows,
    )

    eligible_mask = (
        df_h["state"].isin([STATE_NORMAL, STATE_BEFORE])
        & df_h["future_horizon_valid"]
    )

    supervised_df = df_h.loc[eligible_mask].copy()

    supervised_df["transition_target"] = np.where(
        supervised_df["future_has_during"],
        1,
        0,
    ).astype("int64")

    supervised_df["transition_label"] = np.where(
        supervised_df["transition_target"] == 1,
        f"TO_{STATE_DURING}_WITHIN_{horizon_windows}_WINDOWS",
        f"NO_{STATE_DURING}_WITHIN_{horizon_windows}_WINDOWS",
    )

    supervised_df["scenario_name"] = scenario_name
    supervised_df["scenario_label"] = scenario_name
    supervised_df["target_definition"] = (
        "1 if DURING occurs within H future windows in the same cell; 0 otherwise"
    )

    return supervised_df


def value_counts_dict(series: pd.Series) -> dict:
    """Converte value_counts em dicionário serializável em JSON."""
    return {
        str(k): int(v)
        for k, v in series.value_counts().sort_index().to_dict().items()
    }


def crosstab_to_nested_dict(ct: pd.DataFrame) -> dict:
    """Converte tabela cruzada em dicionário aninhado."""
    result = {}
    for idx in ct.index:
        result[str(idx)] = {
            str(col): int(ct.loc[idx, col])
            for col in ct.columns
        }
    return result


def summarize_scenario(
    cell_id: str,
    scenario_name: str,
    k_before: int,
    horizon_windows: int,
    labeled_df: pd.DataFrame,
    supervised_df: pd.DataFrame,
    effective_episodes_df: pd.DataFrame,
) -> dict:
    """Resume estados, alvo e diagnósticos de circularidade para uma célula/cenário."""
    state_counts = value_counts_dict(labeled_df["state"])
    target_counts = value_counts_dict(supervised_df["transition_target"])

    state_target_ct = pd.crosstab(
        supervised_df["state"],
        supervised_df["transition_target"],
        dropna=False,
    )

    for col in [0, 1]:
        if col not in state_target_ct.columns:
            state_target_ct[col] = 0

    state_target_ct = state_target_ct[[0, 1]]

    normal_target_1 = 0
    before_target_0 = 0
    before_target_1 = 0

    if STATE_NORMAL in state_target_ct.index:
        normal_target_1 = int(state_target_ct.loc[STATE_NORMAL, 1])

    if STATE_BEFORE in state_target_ct.index:
        before_target_0 = int(state_target_ct.loc[STATE_BEFORE, 0])
        before_target_1 = int(state_target_ct.loc[STATE_BEFORE, 1])

    before_indicator = (supervised_df["state"] == STATE_BEFORE).astype("int64")
    target_values = supervised_df["transition_target"].astype("int64")

    if len(supervised_df) > 0:
        target_equals_before_state = bool(
            (before_indicator.values == target_values.values).all()
        )

        positive_states = supervised_df.loc[
            supervised_df["transition_target"] == 1,
            "state",
        ]

        before_targets = supervised_df.loc[
            supervised_df["state"] == STATE_BEFORE,
            "transition_target",
        ]

        all_positive_are_before = bool(
            len(positive_states) > 0
            and (positive_states == STATE_BEFORE).all()
        )

        all_before_are_positive = bool(
            len(before_targets) > 0
            and (before_targets == 1).all()
        )
    else:
        target_equals_before_state = False
        all_positive_are_before = False
        all_before_are_positive = False

    summary = {
        "cell_id": normalize_cell_id(cell_id),
        "scenario_name": scenario_name,
        "scenario_label": scenario_name,
        "k_before": int(k_before),
        "k_after": int(k_before),
        "horizon_windows": int(horizon_windows),
        "horizon_minutes": int(horizon_windows * WINDOW_MINUTES),
        "features_rows": int(len(labeled_df)),
        "effective_episodes": int(len(effective_episodes_df)),
        "effective_during_windows": int((labeled_df["state"] == STATE_DURING).sum()),
        "state_distribution": state_counts,
        "supervised_rows": int(len(supervised_df)),
        "target_distribution": target_counts,
        "state_target_crosstab": crosstab_to_nested_dict(state_target_ct),
        "normal_target_1": int(normal_target_1),
        "before_target_0": int(before_target_0),
        "before_target_1": int(before_target_1),
        "target_equals_before_state": target_equals_before_state,
        "all_positive_are_before": all_positive_are_before,
        "all_before_are_positive": all_before_are_positive,
    }

    return summary


def save_json(path: Path, data: dict) -> None:
    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def write_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, compression="snappy", index=False)


# ─────────────────────────────────────────────────────────────
# BLOCO 4 — Execução por célula
# ─────────────────────────────────────────────────────────────

all_cell_summaries = []
all_scenario_summaries = []
artifact_records = []
primary_labeled_parts = []
primary_supervised_parts = []

for cell_id in ACTIVE_CELLS:
    cell_id = normalize_cell_id(cell_id)
    log("=" * 80)
    log(f"Iniciando célula {cell_id}")

    raw_features, features_source = read_cell_features(cell_id)
    raw_episodes, episodes_source = read_cell_episodes(cell_id)

    df_features = prepare_features_df(raw_features, cell_id)
    df_episodes = prepare_episodes_df(raw_episodes, cell_id)

    bucket_min = int(df_features["bucket_id"].min())
    bucket_max = int(df_features["bucket_id"].max())

    log(f"Features cell={cell_id}: shape={df_features.shape} | fonte={short_path(Path(features_source.split('::')[0]))}")
    log(f"Episódios cell={cell_id}: shape={df_episodes.shape} | fonte={short_path(Path(episodes_source.split('::')[0]))}")
    log(f"Faixa bucket_id cell={cell_id}: {bucket_min}..{bucket_max}")

    effective_episodes = build_effective_episodes(
        episodes_df=df_episodes,
        feature_bucket_min=bucket_min,
        feature_bucket_max=bucket_max,
    )

    if len(effective_episodes) == 0:
        raise RuntimeError(
            f"Nenhum episódio efetivo encontrado na faixa das features para cell={cell_id}."
        )

    expected_during_effective = int(
        effective_episodes["effective_duration_windows"].sum()
    )

    effective_episodes_file = (
        OUT_FEATURES_DIR / f"06_FULL_effective_episodes_cell_{cell_id}.parquet"
    )
    write_parquet(effective_episodes, effective_episodes_file)
    artifact_records.append({
        "artifact": "effective_episodes",
        "cell_id": cell_id,
        "scenario_name": pd.NA,
        "path": str(effective_episodes_file),
    })

    scenario_summaries = []
    labeled_by_k = {}

    for scenario in SCENARIOS:
        scenario_name = scenario["name"]
        k = int(scenario["K"])
        h = int(scenario["H"])

        log(f"Executando cell={cell_id} cenário={scenario_name} (K={k}, H={h})")

        if k not in labeled_by_k:
            labeled_df_k = label_states_for_k(
                features_df=df_features,
                effective_episodes_df=effective_episodes,
                k_before=k,
                k_after=k,
                bucket_min=bucket_min,
                bucket_max=bucket_max,
            )

            actual_during = int((labeled_df_k["state"] == STATE_DURING).sum())

            assert actual_during == expected_during_effective, (
                f"Inconsistência em DURING cell={cell_id}, K={k}: "
                f"esperado={expected_during_effective}, observado={actual_during}"
            )

            labeled_by_k[k] = labeled_df_k

            labeled_k_file = (
                OUT_FEATURES_DIR
                / f"06_FULL_window_5min_labeled_K{k:02d}_cell_{cell_id}.parquet"
            )
            write_parquet(labeled_df_k, labeled_k_file)
            artifact_records.append({
                "artifact": "labeled_by_k",
                "cell_id": cell_id,
                "scenario_name": f"K{k:02d}",
                "path": str(labeled_k_file),
            })

        labeled_df = labeled_by_k[k]

        supervised_df = build_supervised_dataset(
            labeled_df=labeled_df,
            horizon_windows=h,
            scenario_name=scenario_name,
        )

        supervised_file = (
            OUT_FEATURES_DIR
            / f"06_FULL_anticipation_supervised_dataset_K{k:02d}_H{h:02d}_cell_{cell_id}.parquet"
        )
        write_parquet(supervised_df, supervised_file)
        artifact_records.append({
            "artifact": "supervised_scenario",
            "cell_id": cell_id,
            "scenario_name": scenario_name,
            "path": str(supervised_file),
        })

        scenario_summary = summarize_scenario(
            cell_id=cell_id,
            scenario_name=scenario_name,
            k_before=k,
            horizon_windows=h,
            labeled_df=labeled_df,
            supervised_df=supervised_df,
            effective_episodes_df=effective_episodes,
        )
        scenario_summary["features_source"] = features_source
        scenario_summary["episodes_source"] = episodes_source
        scenario_summary["output_supervised_file"] = str(supervised_file)

        scenario_summaries.append(scenario_summary)
        all_scenario_summaries.append(scenario_summary)

        if scenario_name == PRIMARY_SCENARIO_NAME:
            primary_labeled = add_future_horizon_fields(
                labeled_df=labeled_df,
                horizon_windows=h,
            )
            primary_supervised = supervised_df.copy()

            primary_labeled_file = (
                OUT_FEATURES_DIR
                / f"06_FULL_window_5min_labeled_K{k:02d}_H{h:02d}_cell_{cell_id}.parquet"
            )
            primary_supervised_file = (
                OUT_FEATURES_DIR
                / f"06_FULL_anticipation_supervised_dataset_K{k:02d}_H{h:02d}_PRIMARY_cell_{cell_id}.parquet"
            )
            transition_alias_file = (
                OUT_FEATURES_DIR
                / f"06_FULL_transition_dataset_K{k:02d}_H{h:02d}_cell_{cell_id}.parquet"
            )

            write_parquet(primary_labeled, primary_labeled_file)
            write_parquet(primary_supervised, primary_supervised_file)
            write_parquet(primary_supervised, transition_alias_file)

            for artifact, path in [
                ("primary_labeled", primary_labeled_file),
                ("primary_supervised", primary_supervised_file),
                ("primary_transition_alias", transition_alias_file),
            ]:
                artifact_records.append({
                    "artifact": artifact,
                    "cell_id": cell_id,
                    "scenario_name": scenario_name,
                    "path": str(path),
                })

            primary_labeled_parts.append(primary_labeled)
            primary_supervised_parts.append(primary_supervised)

    cell_summary = {
        "cell_id": cell_id,
        "features_source": features_source,
        "episodes_source": episodes_source,
        "features_rows": int(len(df_features)),
        "features_cols": int(df_features.shape[1]),
        "feature_bucket_min": int(bucket_min),
        "feature_bucket_max": int(bucket_max),
        "episodes_original": int(len(df_episodes)),
        "episodes_effective": int(len(effective_episodes)),
        "expected_during_effective_windows": int(expected_during_effective),
        "primary_scenario": PRIMARY_SCENARIO_NAME,
        "scenario_summaries": scenario_summaries,
    }

    all_cell_summaries.append(cell_summary)

    cell_summary_file = (
        OUT_REPORTS_DIR / f"06_FULL_labeling_states_summary_cell_{cell_id}.json"
    )
    save_json(cell_summary_file, cell_summary)
    artifact_records.append({
        "artifact": "cell_summary_json",
        "cell_id": cell_id,
        "scenario_name": pd.NA,
        "path": str(cell_summary_file),
    })

    primary_for_display = [
        s for s in scenario_summaries
        if s["scenario_name"] == PRIMARY_SCENARIO_NAME
    ][0]

    print("\n" + "=" * 80)
    print(f"Resumo principal — cell={cell_id}")
    display(pd.DataFrame([{
        "cell_id": cell_id,
        "K": primary_for_display["k_before"],
        "H": primary_for_display["horizon_windows"],
        "features_rows": primary_for_display["features_rows"],
        "episodes_effective": primary_for_display["effective_episodes"],
        "during_windows": primary_for_display["effective_during_windows"],
        "supervised_rows": primary_for_display["supervised_rows"],
        "target_0": primary_for_display["target_distribution"].get("0", 0),
        "target_1": primary_for_display["target_distribution"].get("1", 0),
        "before_target_0": primary_for_display["before_target_0"],
        "target_equals_before_state": primary_for_display["target_equals_before_state"],
    }]))

# ─────────────────────────────────────────────────────────────
# BLOCO 5 — Consolidação allcells após processamento independente
# ─────────────────────────────────────────────────────────────

assert len(primary_labeled_parts) == len(ACTIVE_CELLS), (
    "Nem todas as células ativas geraram base rotulada principal."
)
assert len(primary_supervised_parts) == len(ACTIVE_CELLS), (
    "Nem todas as células ativas geraram base supervisionada principal."
)

df_labeled_allcells = pd.concat(primary_labeled_parts, ignore_index=True)
df_supervised_allcells = pd.concat(primary_supervised_parts, ignore_index=True)

df_labeled_allcells = df_labeled_allcells.sort_values(
    ["cell_id", "bucket_id"]
).reset_index(drop=True)
df_supervised_allcells = df_supervised_allcells.sort_values(
    ["cell_id", "bucket_id"]
).reset_index(drop=True)

primary_labeled_allcells_file = (
    OUT_FEATURES_DIR
    / f"06_FULL_window_5min_labeled_K{PRIMARY_K:02d}_H{PRIMARY_H:02d}_allcells.parquet"
)
primary_supervised_allcells_file = (
    OUT_FEATURES_DIR
    / f"06_FULL_anticipation_supervised_dataset_K{PRIMARY_K:02d}_H{PRIMARY_H:02d}_allcells.parquet"
)
transition_allcells_alias_file = (
    OUT_FEATURES_DIR
    / f"06_FULL_transition_dataset_K{PRIMARY_K:02d}_H{PRIMARY_H:02d}_allcells.parquet"
)

write_parquet(df_labeled_allcells, primary_labeled_allcells_file)
write_parquet(df_supervised_allcells, primary_supervised_allcells_file)
write_parquet(df_supervised_allcells, transition_allcells_alias_file)

for artifact, path in [
    ("primary_labeled_allcells", primary_labeled_allcells_file),
    ("primary_supervised_allcells", primary_supervised_allcells_file),
    ("primary_transition_alias_allcells", transition_allcells_alias_file),
]:
    artifact_records.append({
        "artifact": artifact,
        "cell_id": "allcells",
        "scenario_name": PRIMARY_SCENARIO_NAME,
        "path": str(path),
    })

# ─────────────────────────────────────────────────────────────
# BLOCO 6 — Diagnósticos consolidados
# ─────────────────────────────────────────────────────────────

summary_table_rows = []

for s in all_scenario_summaries:
    target_0 = int(s["target_distribution"].get("0", 0))
    target_1 = int(s["target_distribution"].get("1", 0))
    total = int(s["supervised_rows"])
    pos_rate = float(target_1 / total) if total > 0 else 0.0

    summary_table_rows.append({
        "cell_id": s["cell_id"],
        "scenario_name": s["scenario_name"],
        "scenario_label": s.get("scenario_label", s["scenario_name"]),
        "K": s["k_before"],
        "H": s["horizon_windows"],
        "horizon_minutes": s["horizon_minutes"],
        "features_rows": s["features_rows"],
        "effective_episodes": s["effective_episodes"],
        "during_windows": s["effective_during_windows"],
        "supervised_rows": total,
        "target_0": target_0,
        "target_1": target_1,
        "positive_rate": pos_rate,
        "normal_target_1": s["normal_target_1"],
        "before_target_0": s["before_target_0"],
        "before_target_1": s["before_target_1"],
        "target_equals_before_state": s["target_equals_before_state"],
        "all_positive_are_before": s["all_positive_are_before"],
        "all_before_are_positive": s["all_before_are_positive"],
    })

summary_table = pd.DataFrame(summary_table_rows).sort_values(
    ["cell_id", "K", "H", "scenario_name"]
).reset_index(drop=True)

summary_table_file = OUT_REPORTS_DIR / "06_FULL_scenario_summary_table.csv"
summary_table.to_csv(summary_table_file, index=False, encoding="utf-8")
artifact_records.append({
    "artifact": "scenario_summary_table_csv",
    "cell_id": "allcells",
    "scenario_name": pd.NA,
    "path": str(summary_table_file),
})

primary_summary_table = summary_table.loc[
    summary_table["scenario_name"] == PRIMARY_SCENARIO_NAME
].copy()

print("\n" + "=" * 80)
print("Resumo consolidado — cenário principal por célula")
display(primary_summary_table)

print("\n" + "=" * 80)
print("Resumo consolidado — todos os cenários")
display(summary_table)

# Validações metodológicas principais.
#
# IMPORTANTE: no ramo FULL, esses achados são diagnósticos por célula.
# Uma célula de borda ou muito esparsa pode não ter BEFORE com target=0,
# por exemplo quando o único episódio começa colado ao bucket_min. Nesse caso,
# a execução não deve invalidar automaticamente as demais células.
# Use STRICT_PRIMARY_CHECKS=1 quando quiser transformar os diagnósticos em erro fatal.
if len(primary_summary_table) != len(ACTIVE_CELLS):
    raise RuntimeError(
        "Resumo principal não contém exatamente uma linha por célula ativa. "
        f"esperado={len(ACTIVE_CELLS)}, observado={len(primary_summary_table)}"
    )

bad_equivalence = primary_summary_table.loc[
    primary_summary_table["target_equals_before_state"] == True  # noqa: E712
].copy()

bad_before_0 = primary_summary_table.loc[
    primary_summary_table["before_target_0"] <= 0
].copy()

# all_positive_are_before só é um problema quando há positivos a validar.
# Células sem target positivo são registradas separadamente como baixa informação
# para a tarefa, mas não quebram a execução desta etapa.
bad_positive_state = primary_summary_table.loc[
    (primary_summary_table["target_1"] > 0)
    & (primary_summary_table["all_positive_are_before"] == False)  # noqa: E712
].copy()

no_positive_target = primary_summary_table.loc[
    primary_summary_table["target_1"] <= 0
].copy()

primary_diagnostic_rows = []

def add_primary_issue(issue_code: str, severity: str, df_issue: pd.DataFrame, message: str) -> None:
    for _, r in df_issue.iterrows():
        primary_diagnostic_rows.append({
            "cell_id": r["cell_id"],
            "scenario_name": r["scenario_name"],
            "scenario_label": r.get("scenario_label", r["scenario_name"]),
            "K": int(r["K"]),
            "H": int(r["H"]),
            "issue_code": issue_code,
            "severity": severity,
            "message": message,
            "features_rows": int(r["features_rows"]),
            "effective_episodes": int(r["effective_episodes"]),
            "during_windows": int(r["during_windows"]),
            "supervised_rows": int(r["supervised_rows"]),
            "target_0": int(r["target_0"]),
            "target_1": int(r["target_1"]),
            "before_target_0": int(r["before_target_0"]),
            "before_target_1": int(r["before_target_1"]),
            "target_equals_before_state": bool(r["target_equals_before_state"]),
            "all_positive_are_before": bool(r["all_positive_are_before"]),
            "all_before_are_positive": bool(r["all_before_are_positive"]),
        })

add_primary_issue(
    issue_code="TARGET_EQUALS_BEFORE_STATE",
    severity="warning_methodological_degeneracy",
    df_issue=bad_equivalence,
    message=(
        "O alvo supervisionado ficou equivalente ao indicador de estado BEFORE nesta célula. "
        "Isso pode ocorrer em célula de borda/esparsa e deve ser tratado downstream como "
        "baixa informação para antecipação, não como falha global do NB06_FULL."
    ),
)

add_primary_issue(
    issue_code="NO_BEFORE_TARGET_ZERO",
    severity="warning_border_cell",
    df_issue=bad_before_0,
    message=(
        "Não há janelas BEFORE com target=0 no cenário principal desta célula. "
        "Isso impede demonstrar localmente o desacoplamento K>H e pode refletir episódio colado "
        "à borda inicial ou baixa densidade de episódios."
    ),
)

add_primary_issue(
    issue_code="POSITIVE_OUTSIDE_BEFORE",
    severity="error_if_strict",
    df_issue=bad_positive_state,
    message=(
        "Há target positivo fora do estado BEFORE apesar de H<=K. "
        "Esse achado sugere inconsistência de rotulagem ou sobreposição inesperada."
    ),
)

add_primary_issue(
    issue_code="NO_POSITIVE_TARGET",
    severity="info_low_signal_cell",
    df_issue=no_positive_target,
    message=(
        "A célula não gerou positivos no alvo de antecipação para o cenário principal. "
        "O artefato é preservado, mas a célula deve ser tratada com cautela nos modelos."
    ),
)

primary_diagnostic_table = pd.DataFrame(primary_diagnostic_rows)
if primary_diagnostic_table.empty:
    primary_diagnostic_table = pd.DataFrame(columns=[
        "cell_id", "scenario_name", "scenario_label", "K", "H", "issue_code",
        "severity", "message", "features_rows", "effective_episodes",
        "during_windows", "supervised_rows", "target_0", "target_1",
        "before_target_0", "before_target_1", "target_equals_before_state",
        "all_positive_are_before", "all_before_are_positive",
    ])

primary_diagnostic_table_file = OUT_REPORTS_DIR / "06_FULL_primary_diagnostic_flags.csv"
primary_diagnostic_table.to_csv(primary_diagnostic_table_file, index=False, encoding="utf-8")
artifact_records.append({
    "artifact": "primary_diagnostic_flags_csv",
    "cell_id": "allcells",
    "scenario_name": PRIMARY_SCENARIO_NAME,
    "path": str(primary_diagnostic_table_file),
})

print("\n" + "=" * 80)
print("Diagnósticos do cenário principal")
if primary_diagnostic_table.empty:
    print("Nenhum diagnóstico metodológico por célula no cenário principal.")
else:
    display(primary_diagnostic_table)
    log(
        "Diagnósticos primários registrados sem abortar execução: "
        f"{primary_diagnostic_table['issue_code'].value_counts().to_dict()}"
    )

strict_problem_table = primary_diagnostic_table.loc[
    primary_diagnostic_table["issue_code"].isin([
        "TARGET_EQUALS_BEFORE_STATE",
        "NO_BEFORE_TARGET_ZERO",
        "POSITIVE_OUTSIDE_BEFORE",
    ])
].copy()

if STRICT_PRIMARY_CHECKS and not strict_problem_table.empty:
    raise RuntimeError(
        "STRICT_PRIMARY_CHECKS=1: diagnósticos metodológicos primários encontrados. "
        f"Arquivo: {primary_diagnostic_table_file}. "
        f"Células/achados: {strict_problem_table[['cell_id', 'issue_code']].to_dict('records')}"
    )

primary_methodological_gate_passed = bool(strict_problem_table.empty)

# ─────────────────────────────────────────────────────────────
# BLOCO 7 — Manifesto SHA-256 e summaries JSON
# ─────────────────────────────────────────────────────────────

summary = {
    "notebook": "NB06_FULL",
    "purpose": "label_states_and_build_anticipation_datasets_by_cell",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "window_minutes": int(WINDOW_MINUTES),
    "active_cells": ACTIVE_CELLS,
    "full_root": str(FULL_ROOT),
    "nb04_full_dir": str(NB04_FULL_DIR),
    "nb05_full_dir": str(NB05_FULL_DIR),
    "nb06_full_dir": str(NB06_FULL_DIR),
    "primary_scenario": PRIMARY_SCENARIO_NAME,
    "primary_scenario_label": PRIMARY_SCENARIO_LABEL,
    "primary_k": int(PRIMARY_K),
    "primary_h": int(PRIMARY_H),
    "primary_horizon_minutes": int(PRIMARY_H * WINDOW_MINUTES),
    "strict_primary_checks": bool(STRICT_PRIMARY_CHECKS),
    "cell_summaries": all_cell_summaries,
    "scenario_summaries": all_scenario_summaries,
    "primary_labeled_allcells_file": str(primary_labeled_allcells_file),
    "primary_supervised_allcells_file": str(primary_supervised_allcells_file),
    "transition_allcells_alias_file": str(transition_allcells_alias_file),
    "summary_table_file": str(summary_table_file),
    "primary_diagnostic_table_file": str(primary_diagnostic_table_file),
    "manifest_file": str(OUT_REPORTS_DIR / "06_FULL_artifact_manifest_sha256.csv"),
    "checks": {
        "cells_processed": int(len(ACTIVE_CELLS)),
        "primary_target_not_equivalent_to_before_all_cells": bool(len(bad_equivalence) == 0),
        "primary_has_before_target_0_all_cells": bool(len(bad_before_0) == 0),
        "primary_all_positive_are_before_when_positive_exists": bool(len(bad_positive_state) == 0),
        "primary_cells_without_positive_target": sorted(no_positive_target["cell_id"].astype(str).tolist()),
        "primary_methodological_gate_passed": bool(primary_methodological_gate_passed),
        "primary_diagnostic_issue_count": int(len(primary_diagnostic_table)),
        "primary_diagnostic_issue_counts": (
            primary_diagnostic_table["issue_code"].value_counts().to_dict()
            if not primary_diagnostic_table.empty else {}
        ),
        "primary_labeled_allcells_rows": int(len(df_labeled_allcells)),
        "primary_supervised_allcells_rows": int(len(df_supervised_allcells)),
    },
}

sensitivity_summary = {
    "notebook": "NB06_FULL",
    "purpose": "horizon_and_state_window_sensitivity_by_cell",
    "interpretation_note": (
        "The legacy K=12,H=12 configuration is retained for comparison. "
        "The revised primary scenario uses K=24,H=12 to decouple the state-labeling "
        "window from the supervised prediction horizon while preserving a 60-minute "
        "prediction horizon. In FULL, this analysis is performed independently per cell."
    ),
    "summary_table": summary_table.to_dict("records"),
    "primary_summary_table": primary_summary_table.to_dict("records"),
    "primary_diagnostic_table": primary_diagnostic_table.to_dict("records"),
}

summary_file = OUT_REPORTS_DIR / "06_FULL_labeling_states_summary.json"
sensitivity_summary_file = OUT_REPORTS_DIR / "06_FULL_horizon_sensitivity_summary.json"
manifest_file = OUT_REPORTS_DIR / "06_FULL_artifact_manifest_sha256.csv"

save_json(summary_file, summary)
save_json(sensitivity_summary_file, sensitivity_summary)

# Summaries entram no manifesto; o próprio manifesto é registrado como
# self_reference, sem SHA-256, para evitar a inconsistência matemática de
# hashear um arquivo que contém uma linha sobre si mesmo.
for artifact, path in [
    ("full_summary_json", summary_file),
    ("full_sensitivity_summary_json", sensitivity_summary_file),
]:
    artifact_records.append({
        "artifact": artifact,
        "cell_id": "allcells",
        "scenario_name": pd.NA,
        "path": str(path),
    })

artifact_df = pd.DataFrame(artifact_records)
artifact_df["path"] = artifact_df["path"].astype(str)
artifact_df["scenario_label"] = artifact_df["scenario_name"]

manifest_rows = []
for _, row in artifact_df.iterrows():
    p = Path(row["path"])
    if not p.exists():
        raise FileNotFoundError(f"Artefato registrado não existe: {p}")

    manifest_rows.append({
        "artifact": row["artifact"],
        "cell_id": row["cell_id"],
        "scenario_name": row["scenario_name"],
        "scenario_label": row["scenario_label"],
        "path": str(p),
        "relative_path": short_path(p),
        "size_bytes": int(p.stat().st_size),
        "sha256": sha256_file(p),
        "hash_status": "ok",
        "hash_note": "",
    })

manifest_rows.append({
    "artifact": "artifact_manifest_sha256_csv",
    "cell_id": "allcells",
    "scenario_name": pd.NA,
    "scenario_label": pd.NA,
    "path": str(manifest_file),
    "relative_path": short_path(manifest_file),
    "size_bytes": pd.NA,
    "sha256": pd.NA,
    "hash_status": "self_reference",
    "hash_note": (
        "Linha intencionalmente sem SHA-256: o manifesto referencia a si mesmo "
        "e qualquer hash calculado antes da escrita final ficaria obsoleto."
    ),
})

manifest_df = pd.DataFrame(manifest_rows).sort_values(
    ["cell_id", "artifact", "scenario_label", "path"],
    na_position="last",
).reset_index(drop=True)
manifest_df.to_csv(manifest_file, index=False, encoding="utf-8")

log(f"Resumo principal salvo: {summary_file}")
log(f"Resumo de sensibilidade salvo: {sensitivity_summary_file}")
log(f"Manifesto SHA-256 salvo: {manifest_file}")

# ─────────────────────────────────────────────────────────────
# BLOCO 8 — Resumo final na tela
# ─────────────────────────────────────────────────────────────

print("\n=== RESUMO FINAL — NB06_FULL ===")
print(f"Células processadas: {ACTIVE_CELLS}")
print(f"Cenário principal: {PRIMARY_SCENARIO_NAME}")
print(f"K principal: {PRIMARY_K}")
print(f"H principal: {PRIMARY_H} janelas ({PRIMARY_H * WINDOW_MINUTES} minutos)")
print(f"Base rotulada allcells: {df_labeled_allcells.shape}")
print(f"Base supervisionada allcells: {df_supervised_allcells.shape}")
print()
print("Artefatos principais:")
print(" -", primary_labeled_allcells_file)
print(" -", primary_supervised_allcells_file)
print(" -", transition_allcells_alias_file)
print(" -", summary_file)
print(" -", sensitivity_summary_file)
print(" -", manifest_file)
print()
print("Checks:")
print(json.dumps(summary["checks"], indent=2, ensure_ascii=False))


Mounted at /content/drive
[Bootstrap] Repositório encontrado: /content/drive/MyDrive/Mestrado/PPCOMP_DM
[Bootstrap] Atualizando repositório (git pull)...
[Bootstrap] CWD = /content/drive/MyDrive/Mestrado/PPCOMP_DM
FULL_ROOT      = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream
NB04_FULL_DIR  = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes
NB05_FULL_DIR  = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/05_FULL_features
NB06_FULL_DIR  = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/06_FULL_labeling_states
OUT_FEATURES   = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/06_FULL_labeling_states/features
OUT_REPORTS    = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/06_FULL_labeling_states/reports
[NB06_FULL_labeling_states] Células ativas: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
[NB06_FULL_labeling_states] Cenário principal: primary_K24_H12
[NB06_FULL_labeling_states] STRICT_PRIMAR

,cell_id,K,H,features_rows,episodes_effective,during_windows,supervised_rows,target_0,target_1,before_target_0,target_equals_before_state
0,a,24,12,8925,191,441,7360,6326,1034,636,False


[NB06_FULL_labeling_states] ================================================================================
[NB06_FULL_labeling_states] Iniciando célula b
[NB06_FULL_labeling_states] Features cell=b: shape=(8925, 47) | fonte=/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/05_FULL_features/cell_b/05_FULL_window_5min_features_TRAIN_M2S_cell_b.parquet
[NB06_FULL_labeling_states] Episódios cell=b: shape=(219, 19) | fonte=/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes/cell_b/04_FULL_episodes_TRAIN_M2S_cell_b.parquet
[NB06_FULL_labeling_states] Faixa bucket_id cell=b: 5..8929
[NB06_FULL_labeling_states] Executando cell=b cenário=legacy_K12_H12 (K=12, H=12)
[NB06_FULL_labeling_states] Executando cell=b cenário=sensitivity_K12_H06 (K=12, H=6)
[NB06_FULL_labeling_states] Executando cell=b cenário=sensitivity_K12_H03 (K=12, H=3)
[NB06_FULL_labeling_states] Executando cell=b cenário=primary_K24_H12 (K=24, H=12)
[NB06_FULL_labeling_states] Executando 

,cell_id,K,H,features_rows,episodes_effective,during_windows,supervised_rows,target_0,target_1,before_target_0,target_equals_before_state
0,b,24,12,8925,219,478,7707,6364,1343,537,False


[NB06_FULL_labeling_states] ================================================================================
[NB06_FULL_labeling_states] Iniciando célula c
[NB06_FULL_labeling_states] Features cell=c: shape=(8925, 47) | fonte=/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/05_FULL_features/cell_c/05_FULL_window_5min_features_TRAIN_M2S_cell_c.parquet
[NB06_FULL_labeling_states] Episódios cell=c: shape=(178, 19) | fonte=/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes/cell_c/04_FULL_episodes_TRAIN_M2S_cell_c.parquet
[NB06_FULL_labeling_states] Faixa bucket_id cell=c: 5..8929
[NB06_FULL_labeling_states] Executando cell=c cenário=legacy_K12_H12 (K=12, H=12)
[NB06_FULL_labeling_states] Executando cell=c cenário=sensitivity_K12_H06 (K=12, H=6)
[NB06_FULL_labeling_states] Executando cell=c cenário=sensitivity_K12_H03 (K=12, H=3)
[NB06_FULL_labeling_states] Executando cell=c cenário=primary_K24_H12 (K=24, H=12)
[NB06_FULL_labeling_states] Executando 

...


Resumo consolidado — cenário principal por célula


,cell_id,scenario_name,scenario_label,K,H,horizon_minutes,features_rows,effective_episodes,during_windows,supervised_rows,target_0,target_1,positive_rate,normal_target_1,before_target_0,before_target_1,target_equals_before_state,all_positive_are_before,all_before_are_positive
5,a,primary_K24_H12,primary_K24_H12,24,12,60,8925,191,441,7360,6326,1034,0.140489,0,636,1034,False,True,False
11,b,primary_K24_H12,primary_K24_H12,24,12,60,8925,219,478,7707,6364,1343,0.174257,0,537,1343,False,True,False
17,c,primary_K24_H12,primary_K24_H12,24,12,60,8925,178,380,7258,6017,1241,0.170984,0,784,1241,False,True,False
23,d,primary_K24_H12,primary_K24_H12,24,12,60,8925,130,413,8288,7781,507,0.061173,0,162,507,False,True,False
29,e,primary_K24_H12,primary_K24_H12,24,12,60,8925,110,245,7737,6889,848,0.109603,0,587,848,False,True,False
35,f,primary_K24_H12,primary_K24_H12,24,12,60,8925,135,247,7169,5889,1280,0.178547,0,924,1280,False,True,False
41,g,primary_K24_H12,primary_K24_H12,24,12,60,8925,295,602,7151,5010,2141,0.299399,0,914,2141,False,True,False
47,h,primary_K24_H12,primary_K24_H12,24,12,60,8925,225,293,6694,4581,2113,0.315656,0,1409,2113,False,True,False



Resumo consolidado — todos os cenários


,cell_id,scenario_name,scenario_label,K,H,horizon_minutes,features_rows,effective_episodes,during_windows,supervised_rows,target_0,target_1,positive_rate,normal_target_1,before_target_0,before_target_1,target_equals_before_state,all_positive_are_before,all_before_are_positive
0,a,sensitivity_K12_H03,sensitivity_K12_H03,12,3,15,8925,191,441,7845,7435,410,0.052263,0,624,410,False,True,False
1,a,sensitivity_K12_H06,sensitivity_K12_H06,12,6,30,8925,191,441,7842,7191,651,0.083015,0,383,651,False,True,False
2,a,legacy_K12_H12,legacy_K12_H12,12,12,60,8925,191,441,7836,6802,1034,0.131955,0,0,1034,True,True,True
3,a,sensitivity_K24_H03,sensitivity_K24_H03,24,3,15,8925,191,441,7369,6959,410,0.055638,0,1260,410,False,True,False
4,a,sensitivity_K24_H06,sensitivity_K24_H06,24,6,30,8925,191,441,7366,6715,651,0.088379,0,1019,651,False,True,False
5,a,primary_K24_H12,primary_K24_H12,24,12,60,8925,191,441,7360,6326,1034,0.140489,0,636,1034,False,True,False
6,b,sensitivity_K12_H03,sensitivity_K12_H03,12,3,15,8925,219,478,7907,7370,537,0.067915,0,806,537,False,True,False
7,b,sensitivity_K12_H06,sensitivity_K12_H06,12,6,30,8925,219,478,7904,7017,887,0.112222,0,456,887,False,True,False
8,b,legacy_K12_H12,legacy_K12_H12,12,12,60,8925,219,478,7898,6555,1343,0.170043,0,0,1343,True,True,True
9,b,sensitivity_K24_H03,sensitivity_K24_H03,24,3,15,8925,219,478,7716,7179,537,0.069596,0,1343,537,False,True,False



Diagnósticos do cenário principal
Nenhum diagnóstico metodológico por célula no cenário principal.
[NB06_FULL_labeling_states] Resumo principal salvo: /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/06_FULL_labeling_states/reports/06_FULL_labeling_states_summary.json
[NB06_FULL_labeling_states] Resumo de sensibilidade salvo: /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/06_FULL_labeling_states/reports/06_FULL_horizon_sensitivity_summary.json
[NB06_FULL_labeling_states] Manifesto SHA-256 salvo: /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/06_FULL_labeling_states/reports/06_FULL_artifact_manifest_sha256.csv

=== RESUMO FINAL — NB06_FULL ===
Células processadas: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
Cenário principal: primary_K24_H12
K principal: 24
H principal: 12 janelas (60 minutos)
Base rotulada allcells: (71400, 62)
Base supervisionada allcells: (59364, 66)

Artefatos principais:
 - /content/drive/MyDrive/Mestrado/04-reports/99_FULL_

## 9. Conclusão da etapa

O NB06_FULL concluiu com sucesso a derivação dos estados operacionais e a construção das bases supervisionadas do experimento FULL. As oito células foram processadas separadamente, preservando a ordem temporal, os episódios herdados do NB04_FULL e os atributos causais produzidos pelo NB05_FULL.

No cenário principal `primary_K24_H12`, com contexto de 24 janelas e horizonte de 12 janelas, equivalente a 60 minutos, a base rotulada consolidada contém **71.400 janelas e 62 colunas**. A base supervisionada consolidada contém **59.364 amostras e 66 colunas**, distribuídas em **48.857 casos negativos** e **10.507 casos positivos**.

A distribuição da base supervisionada no cenário principal foi:

| Célula | Amostras | Positivos | `positive_rate` | `BEFORE,target=0` |
|---|---:|---:|---:|---:|
| a | 7.360 | 1.034 | 14,05% | 636 |
| b | 7.707 | 1.343 | 17,43% | 537 |
| c | 7.258 | 1.241 | 17,10% | 784 |
| d | 8.288 | 507 | 6,12% | 162 |
| e | 7.737 | 848 | 10,96% | 587 |
| f | 7.169 | 1.280 | 17,85% | 924 |
| g | 7.151 | 2.141 | 29,94% | 914 |
| h | 6.694 | 2.113 | 31,57% | 1.409 |

Essas proporções representam a frequência de `target=1` na base supervisionada de cada célula. Elas não correspondem à prevalência de treino, à prevalência do teste fixo ou às distribuições das partições progressivas, que são definidas nas etapas de modelagem.

A etapa preservou **1.483 episódios efetivos** e **3.099 janelas `DURING`**. Por célula, os episódios e as janelas críticas permaneceram em: a=191/441, b=219/478, c=178/380, d=130/413, e=110/245, f=135/247, g=295/602 e h=225/293.

Os diagnósticos metodológicos do cenário principal foram integralmente aprovados. Em todas as células:

- o alvo não é equivalente ao estado `BEFORE`;
- existem observações `BEFORE` com `target=0`;
- todos os casos positivos pertencem à região `BEFORE`;
- nenhuma célula ficou sem casos positivos;
- não houve mistura temporal entre células;
- não foram registradas ocorrências metodológicas no arquivo de diagnóstico.

A presença de **5.953 observações `BEFORE,target=0`** no conjunto consolidado confirma que a formulação não reduz a tarefa à simples identificação do estado `BEFORE`. O modelo deverá distinguir as janelas pré-críticas cuja entrada em `DURING` ocorre dentro do horizonte H daquelas em que o episódio permanece além desse horizonte.

Foram processados seis cenários por célula, totalizando 48 combinações, com horizontes de 15, 30 e 60 minutos e contextos K=12 ou K=24. O cenário principal permaneceu identificado explicitamente como `primary_K24_H12`, enquanto os demais foram preservados como análises de sensibilidade.

Os artefatos individuais e consolidados foram gravados nas estruturas `features/` e `reports/`, incluindo as bases rotulada, supervisionada e de transição, os summaries principal e de sensibilidade, os diagnósticos por célula e o manifesto SHA-256.

Dessa forma, o NB06_FULL está **aprovado** e apto a alimentar as etapas subsequentes do experimento FULL. A etapa fornece uma base supervisionada prospectiva, desagregada por célula e metodologicamente não degenerada, preservando a distinção entre contexto temporal K, horizonte futuro H, estado operacional e variável-alvo.
